# Filter Label Improvement Visualization

This notebook visualizes existing filter code without reimplementing it. The filter algorithms live in `src/label_filters.py` and `src/best_enhancer.py`; this notebook only loads labels, selects a manageable text crop, runs named strategies, and compares the resulting candidate/text labels.

## 0. Imports

Keep this notebook DRY: import the production code and script helpers instead of copying filter logic into notebook cells.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
SCRIPTS = ROOT / 'scripts'
for p in [SRC, SCRIPTS]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from label_filters import STRATEGIES, load_label, run_strategy, visualize_filter_stages
from best_enhancer import enhance as best_enhance, compute_diff_stats as best_diff_stats
from letter_candidates import (
    GreekTemplateMatcher,
    extract_candidates,
    detect_text_lines,
    assign_confidence_tier,
    build_llm_line_context,
    visualize_candidates_overlay,
    visualize_text_lines,
)
from compare_label_filters import find_text_crops, compute_metrics

plt.rcParams['figure.dpi'] = 120
print('Loaded filter registry:', len(STRATEGIES), 'strategies')
print(sorted(STRATEGIES))

## 1. Configuration

Use a crop for interactive visualization. Full-segment enhancement is handled by scripts such as `scripts/enhance_all_labels.py` and `scripts/improve_labels_visual.py`.

In [ ]:
SEG_ID = '20231221180251'
LABEL_PATH = ROOT / 'data' / 'labelled_segments' / SEG_ID / 'ink_labels.tif'

CROP_SIZE = 768
THRESHOLD = 0.40
MANUAL_CROP = None  # set to (y0, y1, x0, x1) to inspect a specific region

# Keep the comparison focused. S9 can be slow; enable it only for small crops.
STRATEGIES_TO_SHOW = [
    'S0_raw',
    'S1_current',
    'S2_median_morph',
    'S5_tv_hysteresis',
    'S7_sato_hysteresis',
    'S11_tv_sato_hysteresis',
]

print(LABEL_PATH)
print('exists:', LABEL_PATH.exists())

## 2. Load Label And Select A Text-Like Crop

Crop selection reuses `find_text_crops` from `scripts/compare_label_filters.py` so the notebook and script inspect the same kind of regions.

In [ ]:
label = load_label(LABEL_PATH)
print(f'shape={label.shape} min={label.min():.3f} max={label.max():.3f} mean={label.mean():.4f}')
print(f'ink coverage @ {THRESHOLD:.2f}: {(label > THRESHOLD).mean() * 100:.2f}%')

if MANUAL_CROP is None:
    y, x = find_text_crops(label, crop=CROP_SIZE, n=1, threshold=0.55)[0]
    CROP = (y, y + CROP_SIZE, x, x + CROP_SIZE)
else:
    CROP = MANUAL_CROP

y0, y1, x0, x1 = CROP
crop = label[y0:y1, x0:x1]
print('crop:', CROP, 'shape:', crop.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(crop, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('raw probability crop')
axes[1].imshow(crop > THRESHOLD, cmap='gray')
axes[1].set_title(f'raw binary @ {THRESHOLD:.2f}')
axes[2].hist(crop.ravel(), bins=64, range=(0, 1), color='steelblue', edgecolor='none')
axes[2].axvspan(0.40, 0.55, color='orange', alpha=0.2, label='ignore band')
axes[2].set_title('crop label distribution')
axes[2].legend()
for ax in axes[:2]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Run Existing Filter Strategies

`run_strategy` calls the registry in `src/label_filters.py`. `best_enhance` calls the currently preferred TV + Sato + hysteresis implementation in `src/best_enhancer.py`.

In [ ]:
outputs = {}
for name in STRATEGIES_TO_SHOW:
    print('running', name)
    outputs[name] = run_strategy(name, crop)

print('running best_enhancer_soft')
outputs['best_enhancer_soft'] = best_enhance(crop, soft_output=True)
print('running best_enhancer_hard')
outputs['best_enhancer_hard'] = best_enhance(crop, soft_output=False)

print('done:', list(outputs))

## 4. Visual Before/After Comparison

In [ ]:
names = list(outputs)
cols = 3
rows = int(np.ceil((len(names) + 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 4.2))
axes = np.atleast_1d(axes).ravel()

axes[0].imshow(crop, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('original')
axes[0].axis('off')

for i, name in enumerate(names, start=1):
    axes[i].imshow(outputs[name], cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(name)
    axes[i].axis('off')

for ax in axes[len(names) + 1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Quantitative Crop Metrics

These are structural proxy metrics, not ground truth accuracy. They help identify whether a filter is suppressing speckles, preserving letter-sized components, and changing ink coverage too aggressively.

In [ ]:
rows = []
all_maps = {'original': crop, **outputs}
for name, arr in all_maps.items():
    m = compute_metrics(arr, binarize_threshold=0.5)
    diff = arr - crop
    rows.append({
        'strategy': name,
        'mean': float(arr.mean()),
        'ink_pct_t40': float((arr > 0.40).mean() * 100),
        'ink_pct_t55': float((arr > 0.55).mean() * 100),
        'mean_abs_diff': float(np.abs(diff).mean()),
        'pct_increased_gt_005': float((diff > 0.05).mean() * 100),
        'pct_decreased_gt_005': float((diff < -0.05).mean() * 100),
        **m,
    })

metrics_df = pd.DataFrame(rows).set_index('strategy')
display(metrics_df.round(4))

## 6. Diff View For Preferred Filter

Use this to inspect whether the chosen filter is cleaning noise, reconnecting strokes, or deleting useful faint ink.

In [ ]:
PREFERRED = 'best_enhancer_soft'
preferred = outputs[PREFERRED]

fig = visualize_filter_stages(crop, preferred, title_suffix=f' - {PREFERRED}')
plt.show()

display(pd.Series(best_diff_stats(crop, preferred)).round(4))

## 7. Candidate/Text Label Improvement

This section shows what the processing does to extracted letter candidates and line contexts. It uses `letter_candidates.py`; no letter matching logic is duplicated here.

In [ ]:
matcher = GreekTemplateMatcher()

def summarize_candidates(prob_map, label, *, threshold=0.40, top_k=3):
    candidates = extract_candidates(prob_map, threshold=threshold, min_area=50, max_area=8000)
    for cand in candidates:
        if assign_confidence_tier(cand) != 'LOW':
            cand.matches = matcher.match(cand.patch, top_k=top_k)
    lines = detect_text_lines(candidates)
    tier_counts = {'HIGH': 0, 'MEDIUM': 0, 'LOW': 0}
    for cand in candidates:
        tier_counts[assign_confidence_tier(cand)] += 1
    return candidates, lines, {'label': label, 'n_candidates': len(candidates), 'n_lines': len(lines), **tier_counts}

cand_raw, lines_raw, summary_raw = summarize_candidates(crop, 'original', threshold=THRESHOLD)
cand_enh, lines_enh, summary_enh = summarize_candidates(preferred, PREFERRED, threshold=THRESHOLD)

summary_df = pd.DataFrame([summary_raw, summary_enh]).set_index('label')
display(summary_df)

## 8. Candidate Overlay Before/After

In [ ]:
fig = visualize_candidates_overlay(crop, cand_raw, figsize=(12, 8), max_labels=250)
fig.suptitle('Original candidates', fontsize=12)
plt.show()

fig = visualize_candidates_overlay(preferred, cand_enh, figsize=(12, 8), max_labels=250)
fig.suptitle(f'{PREFERRED} candidates', fontsize=12)
plt.show()

## 9. Text Line Context Before/After

The text context is what later stages feed to linguistic/LLM reasoning. We compare the first useful line before and after filtering.

In [ ]:
def first_context(lines, width):
    useful = [line for line in lines if len(line.candidates) >= 4]
    if not useful:
        return '(no line with at least 4 candidates)'
    return build_llm_line_context(useful[0], width)

print('--- Original context ---')
print(first_context(lines_raw, crop.shape[1]))
print('\n--- Enhanced context ---')
print(first_context(lines_enh, preferred.shape[1]))

## 10. Full-Segment Processing Command

For full-size labels, use the existing tiled script/module rather than running heavy full-image enhancement inside the notebook.

In [ ]:
print(r'.\venv\Scripts\python.exe scripts\enhance_all_labels.py --seg ' + SEG_ID)
print(r'.\venv\Scripts\python.exe scripts\train_segment_model.py --model unet_v2 --label-root predictions\improved_labels_visual')